# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakhawat-4/my-ml-internship-v2/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# --- Engineered numeric features (log-transform the skewed volume/traffic columns) ---
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

feature_df = df[MODEL_NUMERIC_FEATURES].copy()
for col in MODEL_NUMERIC_FEATURES:
    if feature_df[col].isna().any():
        feature_df[col] = feature_df[col].fillna(feature_df[col].median())

cat_df = pd.get_dummies(df[MODEL_CATEGORICAL_FEATURES].fillna("unknown"), drop_first=True)

feature_vector = pd.concat([feature_df, cat_df], axis=1)
print("One row =", "one page (content_id)")
print("Feature vector shape:", feature_vector.shape)
feature_vector.head()

One row = one page (content_id)
Feature vector shape: (30000, 44)


,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,days_with_impressions,...,word_count_tier_3500+,word_count_tier_<1000,word_count_tier_unknown,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3
0,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,0.0,88,...,False,False,False,True,False,False,False,False,True,False
1,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,0.0,88,...,False,False,False,True,False,False,False,True,False,False
2,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,2.484907,0.0,88,...,True,False,False,True,False,False,False,True,False,False
3,10.0,0.00,0.00,2877.0,19116.0,9.371779,4.077537,4.369448,0.0,88,...,False,False,True,True,False,False,True,False,False,False
4,0.0,0.00,0.00,2803.0,17469.0,9.859588,3.218876,4.983607,0.0,88,...,False,False,False,True,False,False,False,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
notes = [
    ("search_volume", "monthly search demand for the page's primary keyword", "numeric", "none observed", "yes"),
    ("competition", "0-1 keyword competition score", "numeric", "none observed", "yes"),
    ("cpc", "estimated cost-per-click for the keyword", "numeric", "none observed", "yes"),
    ("word_count / char_count", "content length at last crawl", "numeric", "none observed", "yes"),
    ("log_impressions_90d / log_clicks_90d", "log1p of 90-day GSC totals", "numeric (engineered)", "none, log1p handles zeros", "yes"),
    ("log_sessions_90d / log_ai_sessions_90d", "log1p of 90-day GA sessions, incl. AI-referral sessions", "numeric (engineered)", "none, log1p handles zeros", "yes"),
    ("days_with_impressions / days_with_sessions", "count of active days in the 90-day window", "numeric", "none observed", "yes"),
    ("content_age_days", "days since publish", "numeric", "none observed", "yes"),
    ("days_since_last_update", "days since last content edit", "numeric", "none observed", "yes"),
    ("ctr / avg_position", "90-day click-through rate and average SERP position", "numeric", "none observed", "yes"),
    ("engagement_rate / scroll_rate", "GA engagement and scroll-depth rates", "numeric", "none observed", "yes"),
    ("ai_traffic_pct", "share of sessions attributed to AI referrers", "numeric", "none observed", "yes"),
    ("competition_level / content_type / main_intent", "bucketed keyword/content descriptors", "categorical", "filled 'unknown' if missing", "yes"),
    ("age_tier / freshness_tier / word_count_tier", "bucketed versions of the numeric fields above", "categorical", "filled 'unknown' if missing", "yes"),
    ("impression_tier / position_tier", "bucketed visibility descriptors", "categorical", "filled 'unknown' if missing", "yes"),
]

notes_df = pd.DataFrame(notes, columns=["feature", "meaning", "type", "missing handling", "available before prediction moment?"])
notes_df

,feature,meaning,type,missing handling,available before prediction moment?
0,search_volume,monthly search demand for the page's primary k...,numeric,none observed,yes
1,competition,0-1 keyword competition score,numeric,none observed,yes
2,cpc,estimated cost-per-click for the keyword,numeric,none observed,yes
3,word_count / char_count,content length at last crawl,numeric,none observed,yes
4,log_impressions_90d / log_clicks_90d,log1p of 90-day GSC totals,numeric (engineered),"none, log1p handles zeros",yes
5,log_sessions_90d / log_ai_sessions_90d,"log1p of 90-day GA sessions, incl. AI-referral...",numeric (engineered),"none, log1p handles zeros",yes
6,days_with_impressions / days_with_sessions,count of active days in the 90-day window,numeric,none observed,yes
7,content_age_days,days since publish,numeric,none observed,yes
8,days_since_last_update,days since last content edit,numeric,none observed,yes
9,ctr / avg_position,90-day click-through rate and average SERP pos...,numeric,none observed,yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
# The label this project predicts (see scripts/01_prepare_features.py):
#   is_declining_label = 1 if trend_direction == "down" else 0
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("=== TEST 1: is the label's own source column in the feature vector? ===")
leak_candidates = ["trend_direction", "trend_pct"]
for col in leak_candidates:
    in_numeric = col in MODEL_NUMERIC_FEATURES
    in_categorical = col in MODEL_CATEGORICAL_FEATURES
    print(f"{col}: in numeric features = {in_numeric}, in categorical features = {in_categorical}")
print("-> Both are the label's direct source and are correctly EXCLUDED from the feature vector.\n")

print("=== TEST 2: correlation sanity check — does any *used* feature perfectly predict the label? ===")
corr = feature_df.corrwith(df["is_declining_label"]).abs().sort_values(ascending=False)
print(corr.head(8))
print("-> Highest correlation is well below 1.0, so no used feature is a disguised copy of the label.\n")

print("=== TEST 3: future-window check on last-30d / prev-30d columns ===")
# impressions_last_30d and impressions_prev_30d are both drawn from the SAME 90-day
# observation window used to build every other feature (not a future period relative
# to the prediction moment), so including a *ratio* between them is safe — but neither
# column is currently in MODEL_NUMERIC_FEATURES, so this is a non-issue for this model.
print("impressions_last_30d/prev_30d and clicks_last_30d/prev_30d in feature set:",
      any(c in MODEL_NUMERIC_FEATURES for c in
          ["impressions_last_30d", "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d"]))
print("-> Not used as raw features. They are only used upstream to compute trend_pct/trend_direction, i.e. the label itself.")

=== TEST 1: is the label's own source column in the feature vector? ===
trend_direction: in numeric features = False, in categorical features = False
trend_pct: in numeric features = False, in categorical features = False
-> Both are the label's direct source and are correctly EXCLUDED from the feature vector.

=== TEST 2: correlation sanity check — does any *used* feature perfectly predict the label? ===
days_with_impressions     0.190055
log_impressions_90d       0.177473
content_age_days          0.163882
word_count                0.084279
days_since_last_update    0.081383
char_count                0.068681
ctr                       0.061911
avg_position              0.029035
dtype: float64
-> Highest correlation is well below 1.0, so no used feature is a disguised copy of the label.

=== TEST 3: future-window check on last-30d / prev-30d columns ===
impressions_last_30d/prev_30d and clicks_last_30d/prev_30d in feature set: False
-> Not used as raw features. They are only used upst

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
excluded = [
    ("content_id, client_id", "identifiers — carry no predictive signal and client_id must never leak into a public model/report (privacy)"),
    ("trend_direction, trend_pct", "these ARE the label (is_declining_label is derived directly from trend_direction) — including them would be direct target leakage"),
    ("impressions_last_30d, impressions_prev_30d, clicks_last_30d, clicks_prev_30d, sessions_last_30d, sessions_prev_30d", "these are the raw inputs used to compute trend_pct upstream; using them lets the model reconstruct the label through the back door"),
    ("age_tier_order", "a numeric re-encoding of age_tier, already represented by the age_tier categorical dummy — redundant, drop to avoid double-counting one signal"),
    ("char_count_tier", "redundant with word_count_tier and the raw char_count/word_count numeric features"),
    ("provider_used, model_used", "describe which internal tool/model authored or optimized the content, not a real search or engagement signal — keeping them out avoids the model learning a tooling artifact instead of a search pattern"),
]

for col, reason in excluded:
    print(f"- {col}: {reason}")

- content_id, client_id: identifiers — carry no predictive signal and client_id must never leak into a public model/report (privacy)
- trend_direction, trend_pct: these ARE the label (is_declining_label is derived directly from trend_direction) — including them would be direct target leakage
- impressions_last_30d, impressions_prev_30d, clicks_last_30d, clicks_prev_30d, sessions_last_30d, sessions_prev_30d: these are the raw inputs used to compute trend_pct upstream; using them lets the model reconstruct the label through the back door
- age_tier_order: a numeric re-encoding of age_tier, already represented by the age_tier categorical dummy — redundant, drop to avoid double-counting one signal
- char_count_tier: redundant with word_count_tier and the raw char_count/word_count numeric features
- provider_used, model_used: describe which internal tool/model authored or optimized the content, not a real search or engagement signal — keeping them out avoids the model learning a tooling art

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.